# 05 - Excel Dashboard (FR-06)
**Output:** `reports/bank_campaign_dashboard.xlsx`

- [ ] Ekspor data bersih ke sheet `Data Source` (untuk PivotTable)
- [ ] Sheet `KPI`: total kontak, yes, no, response rate
- [ ] Tabel ringkasan + grafik: response rate per pekerjaan, per pendidikan atau jenis kontak
- [ ] Cek angka konsisten dengan hasil SQL

**PivotTable asli:** dibuat manual di Excel (Insert > PivotTable pada sheet `Data Source`). Jangan diklaim otomatis.

In [1]:
# --- Setup path (jalan di VS Code / Jupyter. Untuk Colab, lihat sel opsional di bawah) ---
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "bank-full.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "bank_campaign_clean.csv"
MODELS_DIR = ROOT / "models"
SQL_DIR = ROOT / "sql"
DASHBOARD_PATH = ROOT / "reports" / "bank_campaign_dashboard.xlsx"
import xlsxwriter

In [2]:
# (Opsional) Hanya jika dijalankan di Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/bank-campaign-response-analytics')  # sesuaikan lokasi folder
# Jalankan sel Setup di atas SETELAH baris ini.

In [3]:
df = pd.read_csv(CLEAN_PATH)
# Alternatif: baca dari PostgreSQL agar konsisten dengan notebook 04

In [4]:
required_columns = {
    "age", "job", "education", "contact", "campaign", "poutcome", "month", "y"
}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan di data bersih: {missing_columns}")
if df["y"].isna().any() or not set(df["y"].unique()).issubset({"yes", "no"}):
    raise ValueError("Kolom y harus hanya berisi label 'yes' dan 'no' tanpa nilai kosong.")

# Semua rate dihitung sebagai jumlah yes / total kontak, sama seperti query SQL.
def make_response_summary(data, column, label):
    summary = (
        data.groupby(column, dropna=False, observed=True)
        .agg(
            total_kontak=("y", "size"),
            jumlah_yes=("y", lambda values: values.eq("yes").sum()),
        )
        .reset_index()
        .rename(columns={column: label})
    )
    summary["response_rate_pct"] = (
        summary["jumlah_yes"].div(summary["total_kontak"]).mul(100).round(2)
    )
    return summary

kpi = {
    "total_kontak": int(len(df)),
    "jumlah_yes": int(df["y"].eq("yes").sum()),
    "jumlah_no": int(df["y"].eq("no").sum()),
}
kpi["response_rate_pct"] = round(kpi["jumlah_yes"] / kpi["total_kontak"] * 100, 2)
assert kpi["jumlah_yes"] + kpi["jumlah_no"] == kpi["total_kontak"]

job_summary = make_response_summary(df, "job", "job").sort_values(
    ["response_rate_pct", "total_kontak"], ascending=[False, False]
).reset_index(drop=True)
education_summary = make_response_summary(df, "education", "education").sort_values(
    "response_rate_pct", ascending=False
).reset_index(drop=True)
contact_summary = make_response_summary(df, "contact", "contact").sort_values(
    "response_rate_pct", ascending=False
).reset_index(drop=True)

age_bins = [0, 25, 35, 45, 55, 65, np.inf]
age_labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df_with_age_group = df.assign(
    kelompok_usia=pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)
)
age_summary = make_response_summary(
    df_with_age_group, "kelompok_usia", "kelompok_usia"
)
age_summary["kelompok_usia"] = pd.Categorical(
    age_summary["kelompok_usia"], categories=age_labels, ordered=True
)
age_summary = age_summary.sort_values("kelompok_usia").reset_index(drop=True)

campaign_summary = make_response_summary(df, "campaign", "jumlah_kontak").sort_values(
    "jumlah_kontak"
).reset_index(drop=True)
prior_outcome_summary = make_response_summary(df, "poutcome", "poutcome").sort_values(
    "response_rate_pct", ascending=False
).reset_index(drop=True)
month_summary = make_response_summary(df, "month", "month").sort_values(
    "total_kontak", ascending=False
).reset_index(drop=True)

summary_tables = {
    "By Job": job_summary,
    "By Education": education_summary,
    "By Contact": contact_summary,
    "By Age": age_summary,
    "Campaign": campaign_summary,
    "Prior Outcome": prior_outcome_summary,
    "By Month": month_summary,
}

# Cek lokal terhadap definisi Q0-Q5 pada campaign_analysis.sql; belum membandingkan ke server PostgreSQL.
assert kpi["total_kontak"] == len(df)
assert round(job_summary["response_rate_pct"].iloc[0], 2) == round(
    job_summary["jumlah_yes"].iloc[0] / job_summary["total_kontak"].iloc[0] * 100, 2
)
assert set(age_summary["kelompok_usia"].astype(str)) <= set(age_labels)

DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(DASHBOARD_PATH, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Data Source", index=False)
    for sheet_name, table in summary_tables.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)
    pd.DataFrame().to_excel(writer, sheet_name="KPI", index=False)

    workbook = writer.book
    title_format = workbook.add_format({
        "bold": True, "font_size": 18, "font_color": "#FFFFFF",
        "bg_color": "#174A5B", "align": "left", "valign": "vcenter",
    })
    header_format = workbook.add_format({
        "bold": True, "font_color": "#FFFFFF", "bg_color": "#286B73",
        "text_wrap": True, "valign": "vcenter",
    })
    card_label_format = workbook.add_format({
        "bold": True, "font_color": "#FFFFFF", "bg_color": "#286B73",
        "align": "center", "valign": "vcenter",
    })
    card_value_format = workbook.add_format({
        "bold": True, "font_size": 18, "font_color": "#174A5B",
        "bg_color": "#E6F0ED", "align": "center", "valign": "vcenter",
        "num_format": "#,##0",
    })
    percent_card_format = workbook.add_format({
        "bold": True, "font_size": 18, "font_color": "#174A5B",
        "bg_color": "#E6F0ED", "align": "center", "valign": "vcenter",
        "num_format": '0.00"%"',
    })
    percent_cell_format = workbook.add_format({"num_format": '0.00"%"'})

    source_sheet = writer.sheets["Data Source"]
    source_sheet.freeze_panes(1, 0)
    source_sheet.autofilter(0, 0, len(df), len(df.columns) - 1)
    source_sheet.set_row(0, 30, header_format)
    source_sheet.set_column(0, len(df.columns) - 1, 15)
    source_sheet.set_column(1, 1, 20)
    source_sheet.set_column(3, 3, 18)

    for sheet_name, table in summary_tables.items():
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, len(table), len(table.columns) - 1)
        worksheet.set_row(0, 30, header_format)
        worksheet.set_column(0, 0, 22)
        worksheet.set_column(1, len(table.columns) - 1, 18)
        if "response_rate_pct" in table.columns:
            rate_column = table.columns.get_loc("response_rate_pct")
            worksheet.set_column(rate_column, rate_column, 20, percent_cell_format)

    kpi_sheet = writer.sheets["KPI"]
    kpi_sheet.hide_gridlines(2)
    kpi_sheet.set_column("A:L", 14)
    kpi_sheet.set_row(0, 34)
    kpi_sheet.merge_range("A1:L1", "Bank Campaign Response | Dashboard", title_format)
    kpi_sheet.write("A2", "Ringkasan kampanye telemarketing", workbook.add_format({"italic": True, "font_color": "#52666B"}))

    cards = [
        ("TOTAL KONTAK", kpi["total_kontak"], card_value_format),
        ("BERLANGGANAN (YES)", kpi["jumlah_yes"], card_value_format),
        ("TIDAK BERLANGGANAN (NO)", kpi["jumlah_no"], card_value_format),
        ("RESPONSE RATE", kpi["response_rate_pct"], percent_card_format),
    ]
    for start_column, (label, value, value_format) in zip([0, 3, 6, 9], cards):
        kpi_sheet.merge_range(3, start_column, 3, start_column + 2, label, card_label_format)
        kpi_sheet.merge_range(4, start_column, 5, start_column + 2, value, value_format)

    chart_specs = [
        ("By Job", "job", "Response rate per pekerjaan", "A8"),
        ("By Education", "education", "Response rate per pendidikan", "A26"),
        ("By Contact", "contact", "Response rate per jenis kontak", "A44"),
    ]
    for sheet_name, category_name, title, anchor in chart_specs:
        table = summary_tables[sheet_name]
        rate_column = table.columns.get_loc("response_rate_pct")
        chart = workbook.add_chart({"type": "column"})
        chart.add_series({
            "name": "Response rate",
            "categories": [sheet_name, 1, 0, len(table), 0],
            "values": [sheet_name, 1, rate_column, len(table), rate_column],
            "fill": {"color": "#4C8C72"},
            "border": {"none": True},
            "data_labels": {"value": True, "num_format": '0.0"%"'},
        })
        chart.set_title({"name": title})
        chart.set_x_axis({"name": category_name, "num_font": {"rotation": -45}})
        chart.set_y_axis({"name": "Response rate (%)", "min": 0, "major_gridlines": {"visible": True}})
        chart.set_legend({"none": True})
        chart.set_size({"width": 850, "height": 330})
        kpi_sheet.insert_chart(anchor, chart)

print(f"Dashboard saved: {DASHBOARD_PATH.relative_to(ROOT)}")
print(
    f"KPI lokal: {kpi['total_kontak']:,} kontak | {kpi['jumlah_yes']:,} yes | "
    f"{kpi['jumlah_no']:,} no | response rate {kpi['response_rate_pct']:.2f}%"
)
print("Ringkasan telah dihitung mengikuti definisi query SQL dari CSV; koneksi PostgreSQL tidak dijalankan dari notebook ini.")

Dashboard saved: reports/bank_campaign_dashboard.xlsx
KPI lokal: 45,211 kontak | 5,289 yes | 39,922 no | response rate 11.70%
Ringkasan telah dihitung mengikuti definisi query SQL dari CSV; koneksi PostgreSQL tidak dijalankan dari notebook ini.
